In [4]:
"""
ПАРСИНГ НЕСКОЛЬКИХ СТРАНИЦ ИНТЕРНЕТ-МАГАЗИНА КНИГ
Сайт: http://books.toscrape.com/
"""

# ПОДКЛЮЧАЕМ БИБЛИОТЕКИ
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time # для пауз между запросами

# НАСТРОЙКИ ПАРСИНГА
# создаём шаблон URL для страниц (страницы на сайте идут как catalogue/page-1.html и т.д.)
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
pages_to_scrape = 3  # указываем колько страниц мы хотим спарсить

books_data = []
book_counter = 1

print(f"Начинаем сбор данных с {pages_to_scrape} страниц...")

# ПАРСИНГ КНИГ ПО СТРАНИЦАМ
# ====================================================
for page in range(1, pages_to_scrape + 1):
    url = base_url.format(page)
    print(f"\n Обработка страницы {page}: {url}")

    response = requests.get(url)

    if response.status_code != 200:
        print(f"Не удалось загрузить страницу {page}. Код: {response.status_code}")
        continue

    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')

    # находим все книги на текущей странице
    books = soup.find_all('article', class_='product_pod')
    print(f"Найдено книг на странице: {len(books)}")

    # парсим данные по каждой книгу
    for book in books:
        # название книги хранится в <h3> --> <a>
        # найдём все h3
        h3_tag = book.find('h3')
        if h3_tag and h3_tag.a: # если у h3 есть тег а, оттуда извлечётся текстовое название книги без пробелов
            book_name = h3_tag.a.get('title', h3_tag.a.string).strip()
        else:
            book_name = "Без названия"

        # извлечём цену
        # цена книги хранится в теге <p> class='price_color'
        price_tag = book.find('p', class_='price_color')
        book_price = price_tag.string.strip() if price_tag else "Нет цены"

        # извлечём рейтинг книги
        # рейтинг книги определяется по классам в теге <p> с классом star-rating
        # создадим словарь для преобразования рейтинга из текста (звёздочек) в число
        rating_map = {
            'One': '1/5',
            'Two': '2/5',
            'Three': '3/5',
            'Four': '4/5',
            'Five': '5/5'
        }
        rating_tag = book.find('p', class_='star-rating') # ищем рейтинг у книги
        rating = "Нет рейтинга"
        if rating_tag:
            classes = rating_tag.get('class', []) # если тег рейтинга найден, его элементы добавляются в список
            if len(classes) >= 2: # если у элемента 2 и более классов, берётся второй класс из списка
                rating = rating_map.get(classes[1], "Неизвестно") # ищем соответствие класса рейтинга в словаре

        # сохраняем данные о книге список
        books_data.append({
            'Порядковый номер': book_counter,
            'Название': book_name,
            'Цена': book_price,
            'Рейтинг': rating
        })
        book_counter += 1

    # пауза перед следующим запросом
    time.sleep(1)

# СОХРАНЕНИЕ ДАННЫХ
print(f"\n3. СОХРАНЕНИЕ ДАННЫХ ({len(books_data)} книг)")
df = pd.DataFrame(books_data)
# конвертируем датафрейм в csv файл
df.to_csv('books_data_all_pages.csv', index=False, encoding='utf-8')

print("Данные успешно сохранены в файл 'books_data_all_pages.csv'")
print(df.head())
print(df.tail())

Начинаем сбор данных с 3 страниц...

 Обработка страницы 1: http://books.toscrape.com/catalogue/page-1.html
Найдено книг на странице: 20

 Обработка страницы 2: http://books.toscrape.com/catalogue/page-2.html
Найдено книг на странице: 20

 Обработка страницы 3: http://books.toscrape.com/catalogue/page-3.html
Найдено книг на странице: 20

3. СОХРАНЕНИЕ ДАННЫХ (60 книг)
Данные успешно сохранены в файл 'books_data_all_pages.csv'
   Порядковый номер                               Название    Цена Рейтинг
0                 1                   A Light in the Attic  £51.77     3/5
1                 2                     Tipping the Velvet  £53.74     1/5
2                 3                             Soumission  £50.10     1/5
3                 4                          Sharp Objects  £47.82     4/5
4                 5  Sapiens: A Brief History of Humankind  £54.23     5/5
    Порядковый номер                                           Название  \
55                56        The Torch Is Pass